In [77]:
from google.colab import files
uploaded = files.upload()

Saving TMBD Movie Dataset.csv to TMBD Movie Dataset (12).csv


In [78]:
import pandas as pd

movies=pd.read_csv('TMBD Movie Dataset.csv')
movies.head()


movies.columns

#selecting useful columns
movies=movies[['original_title','overview','genres','keywords','cast','director']]
movies.head()

,original_title,overview,genres,keywords,cast,director
0,Jurassic World,Twenty-two years after the events of Jurassic ...,Action|Adventure|Science Fiction|Thriller,monster|dna|tyrannosaurus rex|velociraptor|island,Chris Pratt|Bryce Dallas Howard|Irrfan Khan|Vi...,Colin Trevorrow
1,Mad Max: Fury Road,An apocalyptic story set in the furthest reach...,Action|Adventure|Science Fiction|Thriller,future|chase|post-apocalyptic|dystopia|australia,Tom Hardy|Charlize Theron|Hugh Keays-Byrne|Nic...,George Miller
2,Insurgent,Beatrice Prior must confront her inner demons ...,Adventure|Science Fiction|Thriller,based on novel|revolution|dystopia|sequel|dyst...,Shailene Woodley|Theo James|Kate Winslet|Ansel...,Robert Schwentke
3,Star Wars: The Force Awakens,Thirty years after defeating the Galactic Empi...,Action|Adventure|Science Fiction|Fantasy,android|spaceship|jedi|space opera|3d,Harrison Ford|Mark Hamill|Carrie Fisher|Adam D...,J.J. Abrams
4,Furious 7,Deckard Shaw seeks revenge against Dominic Tor...,Action|Crime|Thriller,car race|speed|revenge|suspense|car,Vin Diesel|Paul Walker|Jason Statham|Michelle ...,James Wan


In [79]:
#cleaning the data

movies.isnull().sum()
movies=movies.dropna()


movies.rename(columns={'original_title':'title'},inplace=True)

#creating tags

movies['tags']=movies['overview']+" "+movies['genres']+" "+movies['keywords']

#converting to lowercase
movies['tags']= movies['tags'].apply(lambda x:x.lower())

#creating a new dataframe
new_df=movies[['title','tags']]
new_df.head()

,title,tags
0,Jurassic World,twenty-two years after the events of jurassic ...
1,Mad Max: Fury Road,an apocalyptic story set in the furthest reach...
2,Insurgent,beatrice prior must confront her inner demons ...
3,Star Wars: The Force Awakens,thirty years after defeating the galactic empi...
4,Furious 7,deckard shaw seeks revenge against dominic tor...


The actual movie recommendation system.

In [80]:
#vectorization- converting tags to numbers.
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()

vectors.shape

#checking similarities between movies using the vector
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)

similarity.shape#creates a movie to movie similarity matrix

(1287, 1287)

In [81]:
#the recommendation funtion
def recommend(movie):
    movie = movie.lower()

    if movie not in new_df['title'].str.lower().values:
        print("Movie not found!")
        return

    movie_index = new_df[new_df['title'].str.lower() == movie].index[0]
    distances = similarity[movie_index]

    movie_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]

    for i in movie_list:
        print(new_df.iloc[i[0]].title)

In [82]:
#more polished recommendation function
def recommend(movie):
    movie = movie.lower()

    if movie not in new_df['title'].str.lower().values:
        print("\n❌ Movie not found. Please try another name.\n")
        return

    movie_index = new_df[new_df['title'].str.lower() == movie].index[0]
    distances = similarity[movie_index]

    movie_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]

    print("\n🎬 Recommended Movies:\n")
    for i in movie_list:
        print("👉", new_df.iloc[i[0]].title)

In [83]:
#testing
recommend('Inside Out')
recommend('Tangled')
recommend('Harry Potter')
recommend('Slumdog Millionaire')


🎬 Recommended Movies:

👉 Megamind
👉 Memoirs of an Invisible Man
👉 Despicable Me 2
👉 American Beauty
👉 Things We Lost in the Fire

🎬 Recommended Movies:

👉 Enchanted
👉 Star Wars
👉 Toy Story 3
👉 Aladdin
👉 TRON: Legacy

❌ Movie not found. Please try another name.


🎬 Recommended Movies:

👉 Becoming Jane
👉 Gone Girl
👉 Rock of Ages
👉 Solitary Man
👉 Facing the Giants
